# flcourse — quickstart (Colab or local Jupyter)

Every experiment in `experiments/` has a `main(config, overrides)` function, so **one line runs a
whole federated training** — the same as `python experiments/<name>.py --config <file> --set ...`.

Works without a GPU (everything falls back to the CPU).


## 1. Get the code
Either clone your copy of the repository **or** upload `flcourse.zip` to the Colab session (left sidebar → Files) and unzip it.

In [ ]:
import os, sys, subprocess
REPO_URL = ""          # e.g. "https://github.com/<you>/flcourse.git"  (leave empty if you uploaded flcourse.zip)
if os.path.exists("../experiments") and not os.path.exists("experiments"):
    os.chdir("..")                                   # notebook opened from flcourse/notebooks/
elif not os.path.exists("experiments"):
    if not os.path.exists("flcourse/experiments"):
        if REPO_URL:
            subprocess.run(["git", "clone", REPO_URL, "flcourse"], check=True)
        elif os.path.exists("flcourse.zip"):
            subprocess.run(["unzip", "-q", "flcourse.zip"], check=True)
    os.chdir("flcourse")
print("working directory:", os.getcwd())

In [ ]:
# dependencies (torch / torchvision / numpy / matplotlib are pre-installed on Colab)
%pip install -q pyyaml cryptography pycryptodome scikit-learn galois

In [ ]:
!python -m pytest -q

**Assign a GPU** to your **colab workspace** before loading the next line! 

As colab keeps changing its layout every month or so, find out how to do this. Last time i checked its *Runtime → Change runtime type → GPU*

In [ ]:
sys.path.insert(0, os.getcwd())
from flcourse.device import get_device, describe_device
print("device:", describe_device(get_device("auto")))   # cuda if a GPU works, else mps, else cpu

## Some smaller checks



In [ ]:
from IPython.display import Image, display
from experiments import fedavg

summary = fedavg.main("experiments/configs/quick_digits.yaml")       # offline dataset, ~10 s
display(Image(summary["output_dir"] + "/convergence.png"))

In [ ]:
# any config entry can be overridden -- e.g. switch from model uploads to gradient uploads (FedSGD)
summary = fedavg.main("experiments/configs/quick_digits.yaml",
                      overrides=["fl.upload=gradient", "fl.server_lr=0.5", "fl.rounds=40", "client.gradient_batches=-1"])
display(Image(summary["output_dir"] + "/convergence.png"))

In [ ]:
# MNIST, IID vs. strongly non-IID
for cfg in ["fedavg_mnist_iid", "fedavg_mnist_dirichlet"]:
    s = fedavg.main(f"experiments/configs/{cfg}.yaml", overrides=["fl.rounds=10"])
    display(Image(s["output_dir"] + "/partition.png"), Image(s["output_dir"] + "/convergence.png"))

## Robust aggregation

In [ ]:
from experiments import byzantine
for agg in ["mean", "median", "krum"]:
    s = byzantine.main("experiments/configs/byz_ipm_mean.yaml",
                       overrides=[f"aggregator.name={agg}", "fl.rounds=10", f"name=ipm_vs_{agg}"])
    print(agg, "->", round(s["final"]["test_acc"], 3))

In [ ]:
# parameter sweep

from experiments import sweep          # attacks x aggregators heatmap (~7 min on a CPU)
out = sweep.main("experiments/configs/sweep_byzantine_mnist.yaml")
display(Image(out["output_dir"] + "/heatmap.png"))

## DP and secure aggregation



In [ ]:
from experiments import dp_fedavg, secure_aggregation
s = dp_fedavg.main("experiments/configs/dp_central_mnist.yaml")
print("epsilon:", s["epsilon"])
s = secure_aggregation.main("experiments/configs/secaggplus_mnist.yaml", overrides=["fl.rounds=5"])
print(open(s["output_dir"] + "/report.txt").read())
display(Image(s["output_dir"] + "/communication.png"))

In [ ]:
from experiments import secagg_scaling, privacy_attacks
s = secagg_scaling.main("experiments/configs/secagg_scaling.yaml", overrides=["scaling.num_clients=[5,10,20]"])
display(Image(s["output_dir"] + "/scaling.png"))
s = privacy_attacks.main("experiments/configs/privacy_attacks_mnist.yaml")
display(Image(s["output_dir"] + "/gradient_inversion.png"))

---

You can use the fancy methods of the numpy library like vstack which are not implemented as-is in torch. 




In [ ]:
import numpy as np, torch
from flcourse.aggregators import Krum, TrimmedMean
from flcourse.attacks import ALittleIsEnough
from flcourse.bench import measure

honest = np.random.randn(8, 5)
byz = ALittleIsEnough(tau=1.5)(honest)                  # ByzFL interface: honest vectors -> Byzantine vector
X = np.vstack([honest, byz, byz])
print(Krum(f=2)(X))                                     # numpy in -> numpy out
print(TrimmedMean(f=2)(torch.tensor(X)))                # torch in -> torch out
print(measure(torch.zeros(1000)), "bits")               # 32000 bits